In [1]:
import pandas as pd
import requests
import zipfile
import io
import os
from datetime import datetime


# paths
save_dir = "/lakehouse/default/Files/data/raw/nse_trade_inc"
log_file = "/lakehouse/default/Files/data/raw/logs/pipeline_log.csv"

os.makedirs(save_dir, exist_ok=True)

# today's date
today = datetime.today()

date_str = today.strftime("%Y%m%d")
file_date = today.strftime("%d-%m-%Y")

file_path = os.path.join(
    save_dir,
    f"trade_{file_date}.csv"
)

try:

    url = (
        "https://nsearchives.nseindia.com/content/cm/"
        f"BhavCopy_NSE_CM_0_0_0_{date_str}_F_0000.csv.zip"
    )

    headers = {
        "User-Agent": "Mozilla/5.0"
    }

    response = requests.get(
        url,
        headers=headers,
        timeout=20
    )

    if response.status_code != 200:
        raise Exception("No data yet")

    # read bhavcopy
    zip_file = zipfile.ZipFile(
        io.BytesIO(response.content)
    )

    csv_name = zip_file.namelist()[0]

    df = pd.read_csv(
        zip_file.open(csv_name)
    )

    # keep NSE cash market stocks only
    df = df[
        (df["Sgmt"] == "CM") &
        (df["FinInstrmTp"] == "STK")
    ]

    # keep required columns
    df = df[[
        "TradDt",
        "TckrSymb",
        "OpnPric",
        "HghPric",
        "LwPric",
        "ClsPric",
        "TtlTradgVol"
    ]]

    # rename columns
    df.columns = [
        "Date",
        "Symbol",
        "Open",
        "High",
        "Low",
        "Close",
        "Volume"
    ]

    # save file
    df.to_csv(
        file_path,
        index=False
    )

    status = "SUCCESS"
    rows = len(df)
    message = "File saved"

except Exception as e:

    status = "FAILED"
    rows = 0
    message = str(e)


# log row
log_row = pd.DataFrame([{
    "timestamp": datetime.now(),
    "dataset": "nse_trade",
    "status": status,
    "rows": rows,
    "message": message
}])

if os.path.exists(log_file):

    old_log = pd.read_csv(log_file)

    log_df = pd.concat(
        [old_log, log_row],
        ignore_index=True
    )

else:
    log_df = log_row


log_df.to_csv(
    log_file,
    index=False
)

print(status)
print("Rows:", rows)
print(message)

StatementMeta(, 1aafc4aa-d732-4a7b-9efa-86db7eb34434, 3, Finished, Available, Finished, False)

FAILED
Rows: 0
No data yet
